In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/ferdousjannatul/
      /kaggle/input/datasets/ferdousjannatul/sat-ext/
        /kaggle/input/datasets/ferdousjannatul/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/ferdousjannatul/sat-results-a/
        rtdetr-l.pt
        yolo26n.pt
        /kaggle/input/datasets/ferdousjannatul/sat-results-a/data_grouped/
          data.yaml
        /kaggle/input/datasets/ferdousjannatul/sat-results-a/results/
        /kaggle/input/datasets/ferdousjannatul/sat-results-a/results_A/
        /kaggle/input/datasets/ferdousjannatul/sat-results-a/weights/
          grouped__RT-DETR-L__s42_best.pt
          grouped__RT-DETR-L__s43_best.pt
      /kaggle/input/datasets/ferdousjannatul/sat-results-c/
        yolo11m.pt
        yolo26n.pt
        /kaggle/input/datasets/ferdousjannatul/sat-results-c

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/ferdousjannatul/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
import plans, train, infer, dota
utils.require_gpu()
assert utils.install_detection_stack(), "pip install failed - is Internet ON?"
info = dota.build()
for split, m, seed in plans.EXTERNAL:
    rid = f"xdota__{m}__s{seed}"
    if train.find_existing(rid)[0]:
        print("already done:", rid); continue
    _, wt = train.find_existing(f"{split}__{m}__s{seed}")
    if wt is None:
        print("weights not attached:", split, m, seed); continue
    infer.evaluate_run(rid, m, wt, info, modes=dota.modes(), meta=dict(trained_on=split, tested_on="DOTA-v1.0 val sample"))

GPU: {'device': 'cuda:0', 'gpu': 'Tesla T4', 'n_gpu': 2, 'torch': '2.11.0+cu128'}
DOTA: 3015 label files, 1865 with an image
DOTA test sample: 300 images, 24785 boxes (1440 difficult = ignored), fingerprint 5ae49fa86436
  objects by DOTA class (difficult flag): {('large-vehicle', 0): 6027, ('large-vehicle', 1): 290, ('plane', 0): 2373, ('plane', 1): 71, ('ship', 0): 7534, ('ship', 1): 115, ('small-vehicle', 0): 7411, ('small-vehicle', 1): 964}
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  xdota__YOLO11n__s42          standard  AP50=0.2769 AP=0.1044 AR50_micro=0.077 | conf>=0.25: dets=8377 TP=4582 FP=3795 R=0.196 | 2876.8 ms/img
  xdota__YOLO11n__s42          sahi640   AP50=0.6379 AP=0.2936 AR50_micro=0.641 | conf>=0.25: det

2026-10-21 19:56:59,196 - sahi - WARNING - ignoring invalid prediction with bbox: [0, 150.5137481689453, 0, 293.84259033203125] (ultralytics.py:278)


  xdota__YOLOv8m__s42          sahi640   AP50=0.6270 AP=0.3058 AR50_micro=0.713 | conf>=0.25: dets=49233 TP=16415 FP=32818 R=0.703 | 1199.5 ms/img


2026-10-21 20:06:07,036 - sahi - WARNING - ignoring invalid prediction with bbox: [0, 150.5137481689453, 0, 293.84259033203125] (ultralytics.py:278)


  xdota__YOLOv8m__s42          sahi1024  AP50=0.6179 AP=0.3072 AR50_micro=0.335 | conf>=0.25: dets=39060 TP=14599 FP=24461 R=0.625 | 774.5 ms/img
  xdota__RT-DETR-L__s42        standard  AP50=0.3348 AP=0.1292 AR50_micro=0.104 | conf>=0.25: dets=30190 TP=8598 FP=21592 R=0.368 | 203.4 ms/img
  xdota__RT-DETR-L__s42        sahi640   AP50=0.6109 AP=0.2796 AR50_micro=0.769 | conf>=0.25: dets=93347 TP=17522 FP=75825 R=0.751 | 2208.8 ms/img
  xdota__RT-DETR-L__s42        sahi1024  AP50=0.6098 AP=0.2807 AR50_micro=0.321 | conf>=0.25: dets=83179 TP=16423 FP=66756 R=0.703 | 1171.9 ms/img


In [4]:
import shutil
z = shutil.make_archive(os.path.join(cfg.WORK_DIR, "results_" + "08C"), "zip", cfg.RESULTS_DIR)
print("results zip:", z)
shutil.rmtree(os.path.join(cfg.WORK_DIR, "runs"), ignore_errors=True)

results zip: /kaggle/working/results_08C.zip
